# Distributed Spam Detection
Krzysztof Kopel, 2026

In [1]:
import ray
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1" # Only legacy Keras works well with Ray

In [2]:
if ray.is_initialized:
    ray.shutdown()

ray.init()

2026-05-31 19:00:37,436	INFO worker.py:2012 -- Started a local Ray instance.
C:\Users\krzys\Documents\Programy_Python\DistributedMLProject\.venv\Lib\site-packages\ray\_private\worker.py:2051: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


Python version:,3.12.12
Ray version:,2.55.1



## Data loading and preprocessing

In [3]:
import pandas as pd

local_path = r"data\enron_spam_data.csv"

pdf = pd.read_csv(
    local_path,
    on_bad_lines='skip',
    encoding='utf-8',
    low_memory=False
)

pdf = pdf.fillna({"Message": "", "Subject": ""})

df = ray.data.from_pandas(pdf)
df.show(5)

2026-05-31 19:00:51,591	INFO dataset.py:3818 -- Tip: Use `take_batch()` instead of `take() / show()` to return records in pandas or numpy batch format.
2026-05-31 19:00:51,605	INFO logging.py:416 -- Registered dataset logger for dataset dataset_1_0
2026-05-31 19:00:51,624	INFO streaming_executor.py:166 -- Starting execution of Dataset dataset_1_0. Full logs are in C:\Users\krzys\AppData\Local\Temp\ray\session_2026-05-31_19-00-33_565244_24632\logs\ray-data
2026-05-31 19:00:51,625	INFO streaming_executor.py:167 -- Execution plan of Dataset dataset_1_0: InputDataBuffer[Input] -> LimitOperator[limit=5]
2026-05-31 19:00:51,627	WARNING resource_manager.py:169 -- ⚠️  Ray's object store is configured to use only 42.9% of available memory (1.4GiB out of 3.3GiB total). For optimal Ray Data performance, we recommend setting the object store to at least 50% of available memory. You can do this by setting the 'object_store_memory' parameter when calling ray.init() or by setting the RAY_DEFAULT_OBJE

{'Message ID': 0, 'Subject': 'christmas tree farm pictures', 'Message': '', 'Spam/Ham': 'ham', 'Date': '1999-12-10'}
{'Message ID': 1, 'Subject': 'vastar resources , inc .', 'Message': 'gary , production from the high island larger block a - 1 # 2 commenced on\nsaturday at 2 : 00 p . m . at about 6 , 500 gross . carlos expects between 9 , 500 and\n10 , 000 gross for tomorrow . vastar owns 68 % of the gross production .\ngeorge x 3 - 6992\n- - - - - - - - - - - - - - - - - - - - - - forwarded by george weissman / hou / ect on 12 / 13 / 99 10 : 16\nam - - - - - - - - - - - - - - - - - - - - - - - - - - -\ndaren j farmer\n12 / 10 / 99 10 : 38 am\nto : carlos j rodriguez / hou / ect @ ect\ncc : george weissman / hou / ect @ ect , melissa graves / hou / ect @ ect\nsubject : vastar resources , inc .\ncarlos ,\nplease call linda and get everything set up .\ni \' m going to estimate 4 , 500 coming up tomorrow , with a 2 , 000 increase each\nfollowing day based on my conversations with bill fis

In [4]:
def preprocess_batch(batch):
    subject_clean = batch["Subject"].fillna("")
    message_clean = batch["Message"].fillna("")

    batch["Text"] = subject_clean + " " + message_clean

    batch["Spam/Ham"] = batch["Spam/Ham"].map({"ham": 0, "spam": 1})

    return batch

processed_df = df.map_batches(preprocess_batch, batch_format="pandas")

train, test = processed_df.train_test_split(test_size=0.2, shuffle=True)
print("Train set:")
train.show(1)
print("Test set:")
test.show(1)

2026-05-31 19:00:52,364	INFO logging.py:416 -- Registered dataset logger for dataset dataset_4_0
2026-05-31 19:00:52,366	INFO streaming_executor.py:166 -- Starting execution of Dataset dataset_4_0. Full logs are in C:\Users\krzys\AppData\Local\Temp\ray\session_2026-05-31_19-00-33_565244_24632\logs\ray-data
2026-05-31 19:00:52,367	INFO streaming_executor.py:167 -- Execution plan of Dataset dataset_4_0: InputDataBuffer[Input] -> AllToAllOperator[MapBatches(preprocess_batch)->RandomShuffle]
2026-05-31 19:00:52,373	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_4_0 =======
2026-05-31 19:00:52,375	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-05-31 19:00:52,376	INFO logging_progress.py:227 -- Active & requested resources: 0/12 CPU, 0.0B/716.0MiB object store
2026-05-31 19:00:52,376	INFO logging_progress.py:181 -- 
2026-05-31 19:00:52,376	INFO logging_progress.py:231 -- MapBatches(preprocess_batch)->RandomShuffle: 0/1
2026-05-31 19:00:52,377	INFO logging_progress

Train set:
{'Message ID': 9537, 'Subject': 'your membership community charset = iso - 8859 - 1', 'Message': 'your membership community & commentary ( july 6 , 2001 )\nit \' s all about making money\ninformation to provide you with the absolute\nbest low and no cost ways of providing traffic\nto your site , helping you to capitalize on the power and potential the web brings to every net - preneur .\n- - - this issue contains sites who will trade links with you ! - - -\n- - - - - - - - - - - - -\nin this issue\n- - - - - - - - - - - - -\ninternet success through simplicity\nmember showcase\nwin a free ad in community & commentary\n| | | = - = - = - = - = - = - = - = - = - = - = - = - = - = - = - = - = - = > >\ntoday \' s special announcement :\n| | | = - = - = - = - = - = - = - = - = - = - = - = - = - = - = - = - = - = > >\nwe can help you become an internet service provider within 7 days or we will give you $ 100 . 00 ! !\nclick here\nwe have already signed 300 isps on a 4 year contract

## Training a model

In [5]:
import tensorflow as tf

def build_model() -> tf.keras.Model:
    inputs = tf.keras.Input(shape=(200,), dtype=tf.int32)
    x = tf.keras.layers.Embedding(input_dim=10000, output_dim=32)(inputs)
    x = tf.keras.layers.GlobalAveragePooling1D()(x)
    x = tf.keras.layers.Dense(100, activation="relu")(x)
    outputs = tf.keras.layers.Dense(1, activation="sigmoid")(x)
    return tf.keras.Model(inputs=inputs, outputs=outputs)

In [19]:
from ray.train.tensorflow.keras import ReportCheckpointCallback

def training_loop(config: dict):
    tf.keras.backend.clear_session()
    strategy = tf.distribute.MultiWorkerMirroredStrategy()

    batch_size = config.get("batch_size", 32)
    epochs = config.get("epochs", 4)
    X_global = config.get("X_global")
    y_global = config.get("y_global")
    X_val = config.get("X_val")
    y_val = config.get("y_val")

    with strategy.scope():
        model = build_model()
        model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

    model.fit(X_global, y_global, validation_data=(X_val, y_val), batch_size=batch_size, epochs=epochs, callbacks=[ReportCheckpointCallback()])

    ray.train.report(metrics=model.evaluate(X_val, y_val, batch_size=batch_size, return_dict=True))


In [18]:
global_vectorizer = tf.keras.layers.TextVectorization(max_tokens=10000, output_sequence_length=200)
global_vectorizer.adapt(train.to_pandas()["Text"])

X_global = global_vectorizer(train.to_pandas()["Text"].astype(object)).numpy()
y_global = train.to_pandas()["Spam/Ham"].values
X_val = global_vectorizer(test.to_pandas()["Text"].astype(object)).numpy()
y_val = test.to_pandas()["Spam/Ham"].values

2026-05-31 19:46:33,820	INFO logging.py:416 -- Registered dataset logger for dataset dataset_6_21
2026-05-31 19:46:33,829	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_6_21 =======
2026-05-31 19:46:33,831	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-05-31 19:46:33,831	INFO logging_progress.py:227 -- Active & requested resources: 0/12 CPU, 0.0B/716.0MiB object store
2026-05-31 19:46:33,831	INFO logging_progress.py:192 -- =============================================
2026-05-31 19:46:33,840	INFO streaming_executor.py:294 -- ✔️  Dataset dataset_6_21 execution finished in 0.03 seconds
2026-05-31 19:46:35,145	INFO logging.py:416 -- Registered dataset logger for dataset dataset_6_22
2026-05-31 19:46:35,153	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_6_22 =======
2026-05-31 19:46:35,155	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-05-31 19:46:35,155	INFO logging_progress.py:227 -- Active & requested resources: 0/12 CPU, 0.0B/716.

In [21]:
from ray.train.tensorflow import TensorflowTrainer

scaling_config = ray.train.ScalingConfig(num_workers=2, use_gpu=False)

trainer = TensorflowTrainer(
    train_loop_per_worker=training_loop,
    train_loop_config={"batch_size": 32, "epochs": 5, "X_global": X_global, "y_global": y_global, "X_val": X_val, "y_val": y_val},
    scaling_config=scaling_config
)
results = trainer.fit()

(TrainController pid=21304) WARNING: All log messages before absl::InitializeLog() is called are written to STDERR
(TrainController pid=21304) I0000 00:00:1780249897.112193   24636 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
(TrainController pid=21304) WARNING: All log messages before absl::InitializeLog() is called are written to STDERR
(TrainController pid=21304) I0000 00:00:1780249898.565556   24636 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
(TrainController pid=21304) WARNING:tensorflow:From C:\Users\krzys\Documents\Programy_Python\DistributedMLProject\.venv\Lib\site-packages\tf_keras\src\l

(RayTrainWorker pid=19800) Epoch 1/5


(RayTrainWorker pid=19800) INFO:tensorflow:Collective all_reduce tensors: 1 all_reduces, num_devices = 1, group_size = 2, implementation = CommunicationImplementation.AUTO, num_packs = 1
(RayTrainWorker pid=19800) Collective all_reduce tensors: 1 all_reduces, num_devices = 1, group_size = 2, implementation = CommunicationImplementation.AUTO, num_packs = 1
(RayTrainWorker pid=19800) WARNING:tensorflow:From C:\Users\krzys\Documents\Programy_Python\DistributedMLProject\.venv\Lib\site-packages\tf_keras\src\utils\tf_utils.py:492: The name tf.ragged.RaggedTensorValue is deprecated. Please use tf.compat.v1.ragged.RaggedTensorValue instead.
(RayTrainWorker pid=19800) 
(RayTrainWorker pid=19800) From C:\Users\krzys\Documents\Programy_Python\DistributedMLProject\.venv\Lib\site-packages\tf_keras\src\utils\tf_utils.py:492: The name tf.ragged.RaggedTensorValue is deprecated. Please use tf.compat.v1.ragged.RaggedTensorValue instead.
(RayTrainWorker pid=19800) 
(RayTrainWorker pid=19800) INFO:tensorf

(RayTrainWorker pid=19800) 
(RayTrainWorker pid=19800)   1/843 [..............................] - ETA: 10:11 - loss: 1.3881 - accuracy: 0.9375
(RayTrainWorker pid=19800)   9/843 [..............................] - ETA: 5s - loss: 1.3874 - accuracy: 0.8542   
(RayTrainWorker pid=32208) 
(RayTrainWorker pid=19800) 
(RayTrainWorker pid=32208) 
(RayTrainWorker pid=19800) 
(RayTrainWorker pid=19800)  30/843 [>.............................] - ETA: 5s - loss: 1.3803 - accuracy: 1.2417
(RayTrainWorker pid=32208) 
(RayTrainWorker pid=19800) 
(RayTrainWorker pid=32208) 
(RayTrainWorker pid=19800) 
(RayTrainWorker pid=19800)  60/843 [=>............................] - ETA: 5s - loss: 1.3572 - accuracy: 1.3354
(RayTrainWorker pid=32208) 
(RayTrainWorker pid=19800) 
(RayTrainWorker pid=32208) 
(RayTrainWorker pid=19800) 
(RayTrainWorker pid=19800)  90/843 [==>...........................] - ETA: 5s - loss: 1.3030 - accuracy: 1.4181
(RayTrainWorker pid=32208) 
(RayTrainWorker pid=19800) 
(RayTrainWorke

(PlacementGroupCleaner pid=26836) Failed to query Ray Train Controller actor state. State API may be temporarily unavailable. Continuing to monitor.
(RayTrainWorker pid=32208) WARNING: All log messages before absl::InitializeLog() is called are written to STDERR [repeated 3x across cluster]
(RayTrainWorker pid=32208) I0000 00:00:1780249920.469900   26124 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`. [repeated 3x across cluster]


...

(PlacementGroupCleaner pid=26836) Failed to query Ray Train Controller actor state. State API may be temporarily unavailable. Continuing to monitor.
(RayTrainWorker pid=19800) Checkpoint successfully created at: Checkpoint(filesystem=local, path=C:/Users/krzys/ray_results/ray_train_run-2026-05-31_19-51-32/checkpoint_2026-05-31_19-52-43.721091)
(RayTrainWorker pid=19800) Reporting training result 5: TrainingReport(checkpoint=Checkpoint(filesystem=local, path=C:/Users/krzys/ray_results/ray_train_run-2026-05-31_19-51-32/checkpoint_2026-05-31_19-52-43.721091), metrics={'loss': 0.010015110485255718, 'accuracy': 0.9967002868652344, 'val_loss': 0.03778676688671112, 'val_accuracy': 0.9900652170181274}, validation=False)


(RayTrainWorker pid=19800) 
(RayTrainWorker pid=19800) 843/843 [==============================] - 9s 10ms/step - loss: 0.0100 - accuracy: 0.9967 - val_loss: 0.0378 - val_accuracy: 0.9901
(RayTrainWorker pid=32208) 332/843 [==========>...................] - ETA: 4s - loss: 0.0213 - accuracy: 1.9934 [repeated 6x across cluster]
(RayTrainWorker pid=32208) 360/843 [===========>..................] - ETA: 4s - loss: 0.0210 - accuracy: 1.9934 [repeated 6x across cluster]
(RayTrainWorker pid=32208) 392/843 [============>.................] - ETA: 3s - loss: 0.0213 - accuracy: 1.9931 [repeated 8x across cluster]
(RayTrainWorker pid=32208) 420/843 [=============>................] - ETA: 3s - loss: 0.0209 - accuracy: 1.9933 [repeated 10x across cluster]
(RayTrainWorker pid=32208) 
(RayTrainWorker pid=19800) 
(RayTrainWorker pid=32208) 
(RayTrainWorker pid=19800) 
(RayTrainWorker pid=32208) 
(RayTrainWorker pid=19800) 
(RayTrainWorker pid=32208) 448/843 [==============>...............] - ETA: 3s - 

(RayTrainWorker pid=19800) Reporting training result 6: TrainingReport(checkpoint=None, metrics={'loss': 0.03778676688671112, 'accuracy': 0.9900652170181274}, validation=False)


(RayTrainWorker pid=19800) 
(RayTrainWorker pid=19800) 211/211 [==============================] - 1s 5ms/step - loss: 0.0378 - accuracy: 0.9901
(RayTrainWorker pid=32208) 


(PlacementGroupCleaner pid=26836) Failed to query Ray Train Controller actor state. State API may be temporarily unavailable. Continuing to monitor.
(PlacementGroupCleaner pid=26836) Failed to query Ray Train Controller actor state. State API may be temporarily unavailable. Continuing to monitor.
(PlacementGroupCleaner pid=26836) Failed to query Ray Train Controller actor state. State API may be temporarily unavailable. Continuing to monitor.
(raylet) Stack (most recent call first):
(raylet)   File "<frozen importlib._bootstrap_external>", line 757 in _compile_bytecode
(raylet)   File "<frozen importlib._bootstrap_external>", line 1128 in get_code
(raylet)   File "<frozen importlib._bootstrap_external>", line 995 in exec_module
(raylet)   File "<frozen importlib._bootstrap>", line 935 in _load_unlocked
(raylet)   File "<frozen importlib._bootstrap>", line 1331 in _find_and_load_unlocked
(raylet)   File "<frozen importlib._bootstrap>", line 1360 in _find_and_load
(raylet)   File "C:\Use

### Evaluation

In [22]:
print(f"Final results: {results.metrics}")

Final results: {'loss': 0.010015110485255718, 'accuracy': 0.9967002868652344, 'val_loss': 0.03778676688671112, 'val_accuracy': 0.9900652170181274}


Results seem satisfiying, though I will try to tune one of the hyperparameters.